# Fine-tune SmolVLA on the cube-rotation sim demos (stretch goal)

What this trains: `lerobot/smolvla_base` fine-tuned on scripted robosuite demos that replay **my retargeted hand
trajectories** (+60 deg rotation bin, random cube start poses). The policy imitates that scripted controller; it is
a small fine-tune, not a general VLA result.

Inputs: agentview (`observation.images.camera1`) and wrist (`observation.images.camera2`) 256x256 images, a 6-D
state and one fixed instruction. Output: 5-D delta end-effector action (see `src/vla_format.py`).

GPU: written for a Colab GPU runtime. Upstream's notebook says 20k steps at batch 64 took ~5 h on an A100; this
notebook starts with 1k steps. Use batch 32 on a 16 GB T4/L4; lower it on smaller cards. Checked against lerobot 0.6.1.

In [ ]:
!pip install -q "lerobot[smolvla,dataset]==0.6.1"
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Get the dataset
Made locally with `~/.venvs/lerobot/bin/python -m src.to_lerobot` (folder `data/lerobot/cube_rotate_sim`).
Zip it, put it on Google Drive, and unzip it here. Or push it to the Hub and set `--dataset.repo_id` to that repo
instead of `--dataset.root`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!unzip -q -o /content/drive/MyDrive/cube_rotate_sim.zip -d /content/
!ls /content/cube_rotate_sim/meta

## Train: first 1k steps, then check the loss
Stop here and report if the loss is not going down after 1k steps.

In [ ]:
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=local/cube_rotate_sim \
  --dataset.root=/content/cube_rotate_sim \
  --batch_size=32 --steps=1000 --save_freq=1000 --log_freq=50 \
  --output_dir=/content/outputs/smolvla_cube --job_name=smolvla_cube \
  --policy.device=cuda --policy.push_to_hub=false --wandb.enable=false 2>&1 | tee /content/train_1k.log

In [ ]:
import re
import matplotlib.pyplot as plt
log = open("/content/train_1k.log").read()
steps = [int(s.replace("K", "000")) for s in re.findall(r"step:(\d+K?)", log)]
loss = [float(x) for x in re.findall(r"loss:([0-9.]+)", log)]
n = min(len(steps), len(loss))
plt.plot(steps[:n], loss[:n]); plt.xlabel("step"); plt.ylabel("loss"); plt.title("SmolVLA fine-tune, first 1k steps"); plt.show()

## Continue to 5k steps (only if the loss went down)

In [ ]:
!lerobot-train \
  --config_path=/content/outputs/smolvla_cube/checkpoints/last/pretrained_model/train_config.json \
  --resume=true --steps=5000 --save_freq=1000 2>&1 | tee /content/train_5k.log
!cp -r /content/outputs/smolvla_cube/checkpoints/last/pretrained_model /content/drive/MyDrive/smolvla_cube_last

## Local equivalent (lerobot venv, needs a CUDA GPU with enough memory)
```bash
~/.venvs/lerobot/bin/lerobot-train --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=local/cube_rotate_sim --dataset.root=data/lerobot/cube_rotate_sim \
  --batch_size=8 --steps=1000 --save_freq=1000 --log_freq=50 \
  --output_dir=outputs/vla/smolvla_cube --job_name=smolvla_cube \
  --policy.device=cuda --policy.push_to_hub=false --wandb.enable=false
```
Evaluate closed-loop with `python -m src.eval_vla --policy <checkpoint>/pretrained_model` (20 random cube poses,
lifted / placed / yaw / overall, compared with the scripted replay). That needs robosuite next to lerobot, see
`requirements-vla.txt`.